In [1]:
!pip uninstall -q -y transformers
!pip uninstall -q -y accelerate
!pip install -q transformers==4.27.3 accelerate==0.11.0 cryptpandas nltk lightning==2.0.8

In [1]:
cd ..

/workspace/llm-graph-construction


/opt/conda/lib/python3.10/site-packages/IPython/core/magics/osm.py:417: UserWarning: using dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


In [2]:
import torch
from torch import nn
from transformers import AutoTokenizer, AutoModel


/opt/conda/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
from training.train_and_evaluate_relation_extraction import load_stored_dataset_combination_graph

/workspace/llm-graph-construction


In [4]:
# Code injection:
from custom_datasets.dataframe_dataset import DFDataset
def load_stored_dataset_combination_graph(balanced=True, dataset="i2b2"):
    dataset_train = DFDataset()
    dataset_train.load("pregenerated/"+dataset+"_dataset_train_rawkg.pt")
    dataset_val = DFDataset()
    dataset_val.load("pregenerated/"+dataset+"_dataset_val_rawkg.pt")
    dataset_test = DFDataset()
    dataset_test.load("pregenerated/"+dataset+"_dataset_test_rawkg.pt")

    if balanced:
        dataset_train.oversample_pregenerated()
        dataset_val.oversample_pregenerated()

    return dataset_train, dataset_val, dataset_test

In [5]:
dataset_train, dataset_val, dataset_test_ub = load_stored_dataset_combination_graph(balanced=True, dataset="i2b2")
number_of_relations = 3

In [6]:
def collate_function(examples):
    loader = DataLoader(examples, batch_size=len(examples))
    batch = next(iter(loader))
    return {"data": batch, "labels": batch.y}

In [7]:
from torch_geometric.data import DataLoader
batch = collate_function(dataset_train.generated[:2])

In [8]:
batch

{'data': DataBatch(x=[254, 768], edge_index=[2, 1413], edge_attr=[1413, 775], y=[2], edge_type=[1413], event1_index=[2], event2_index=[2], text_relations=[2], text=[2], event1_start=[2], event1_end=[2], event2_start=[2], event2_end=[2], batch=[254], ptr=[3]),
 'labels': tensor([0, 0])}

In [ ]:
# Priprava: to bi bilo v init delu
device = "cuda:0" if torch.cuda.is_available() else "cpu"
device = "cpu"
modelcard = 'plenz/GLM-t5-small'
model = AutoModel.from_pretrained(modelcard, trust_remote_code=True, revision='main')
tokenizer = AutoTokenizer.from_pretrained(modelcard, use_fast=True)
model.to(device)

In [ ]:
# inferenca
device = "cuda:0" if torch.cuda.is_available() else "cpu"
device = "cpu"
graphs = batch["data"]
how = "global"
inputs = []
for g in range(len(graphs["text_relations"])):
    graph = model.data_processor.encode_graph(tokenizer=tokenizer, g=graphs["text_relations"][g], text=graphs["text"][g], how=how)
    inputs.append(graph)
model_inputs = model.data_processor.to_batch(data_instances=inputs, tokenizer=tokenizer, max_seq_len=None, device=device)

In [ ]:
inputs[0].indices

In [ ]:
outputs = model(**model_inputs)

In [ ]:
for i in range(len(graphs["text_relations"])):
    event1 = graphs["text"][i][graphs["event1_start"][i]:graphs["event1_end"][i]]
    event2 = graphs["text"][i][graphs["event2_start"][i]:graphs["event2_end"][i]]
    embedding1 = model.data_processor.get_embedding(sequence_embedding=outputs.last_hidden_state[i], indices=inputs[i].indices, concept=event1, embedding_aggregation='seq')
    embedding2 = model.data_processor.get_embedding(sequence_embedding=outputs.last_hidden_state[i], indices=inputs[i].indices, concept=event2, embedding_aggregation='seq')
    print(embedding1)
    print(embedding2)
    break


In [16]:
import torch
from torch import nn
from transformers import AutoTokenizer, AutoModel

class GraphLanguageModel(nn.Module):
    def __init__(self, number_of_relations=3, combine_embeddings=True):
        super(GraphLanguageModel, self).__init__()
        # self.device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
        self.device = "cpu" # torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
        self.number_of_relations = number_of_relations
        modelcard = 'plenz/GLM-t5-small'
        model_output_size = 256 # t5-small size
        self.model = AutoModel.from_pretrained(modelcard, trust_remote_code=True, revision='main')
        self.model.to(self.device)
        self.tokenizer = AutoTokenizer.from_pretrained(modelcard)
        self.mode = "global"  # global or local

        self.linear = nn.Linear(model_output_size * 2, self.number_of_relations).double()
        self.criterion = nn.CrossEntropyLoss()
        self.softmax = nn.Softmax(dim=1)

    def forward(self, data, labels):
        # data.to(self.device)

        inputs = []
        for i in range(len(data["text_relations"])):
            graph = self.model.data_processor.encode_graph(
                tokenizer=self.tokenizer, g=data["text_relations"][i],
                             text=data["text"][i], how=self.mode)
            inputs.append(graph)
        model_inputs = self.model.data_processor.to_batch(
            data_instances=inputs, tokenizer=self.tokenizer, max_seq_len=None,
                                                     device=self.device)
        outputs = self.model(**model_inputs)
        model_output = []
        for i in range(len(data["text_relations"])):
            event1 = data["text"][i][data["event1_start"][i]:data["event1_end"][i]]
            event2 = data["text"][i][data["event2_start"][i]:data["event2_end"][i]]
            embedding1 = self.model.data_processor.get_embedding(sequence_embedding=outputs.last_hidden_state[i],
                                                            indices=inputs[i].indices, concept=event1,
                                                            embedding_aggregation='mean')
            embedding2 = self.model.data_processor.get_embedding(sequence_embedding=outputs.last_hidden_state[i],
                                                            indices=inputs[i].indices, concept=event2,
                                                            embedding_aggregation='mean')
            print(embedding1.size())
            print(embedding1.size())
            output = torch.cat((embedding1, embedding2))
            model_output.append(output)

        print(model_output)
        print(self.linear)
        x = torch.stack(model_output).double()
        print(x.size())
        x = self.linear(x)
        return self.softmax(x)

In [17]:
model = GraphLanguageModel()

In [18]:
model(batch["data"], batch["labels"])

torch.Size([1, 512])
torch.Size([1, 512])
torch.Size([1, 512])
torch.Size([1, 512])
[tensor([[ 0.1068, -0.0185,  0.1658,  ..., -0.0164, -0.2968,  0.1325],
        [-0.0675, -0.1654,  0.1735,  ..., -0.0491, -0.0175,  0.2697]],
       grad_fn=<CatBackward0>), tensor([[-0.0056, -0.0015,  0.1556,  ..., -0.0048, -0.2209,  0.1438],
        [ 0.1258, -0.2542,  0.1750,  ...,  0.0750, -0.2877, -0.0623]],
       grad_fn=<CatBackward0>)]
Linear(in_features=512, out_features=3, bias=True)
torch.Size([2, 2, 512])


tensor([[[0.4869, 0.5223, 0.4703],
         [0.5131, 0.4777, 0.5297]],

        [[0.5408, 0.5726, 0.4811],
         [0.4592, 0.4274, 0.5189]]], dtype=torch.float64,
       grad_fn=<SoftmaxBackward0>)